# load the csv data

In [1]:
import pandas as pd

# Load the dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
df = pd.read_csv(url)

# Check first few rows
df.head()


,Customer,ST,GENDER,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Vehicle Class,Total Claim Amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


# 1. clean column names

In [2]:
# Standardize column names
df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
df = df.rename(columns={"st": "state", "gender": "gender"})  # Already lowercased

# Verify
df.columns


Index(['customer', 'state', 'gender', 'education', 'customer_lifetime_value',
       'income', 'monthly_premium_auto', 'number_of_open_complaints',
       'policy_type', 'vehicle_class', 'total_claim_amount'],
      dtype='object')

# 2. cleaning invalid values 

In [3]:
# Gender
df['gender'] = df['gender'].str.upper().replace({'FEMALE':'F','MALE':'M'})

# Education
df['education'] = df['education'].replace({'Bachelors':'Bachelor'})

# Customer Lifetime Value
df['customer_lifetime_value'] = df['customer_lifetime_value'].str.replace('%','').astype(float)

# Vehicle class
df['vehicle_class'] = df['vehicle_class'].replace({
    'Sports Car':'Luxury',
    'Luxury SUV':'Luxury',
    'Luxury Car':'Luxury'
})

# States mapping
state_map = {'AZ':'Arizona', 'Cali':'California', 'WA':'Washington'}
df['state'] = df['state'].replace(state_map)


# 3. correct data types

In [5]:
# Number of open complaints
def extract_middle(value):
    if pd.isna(value):  # if the value is NaN
        return 0  # or choose another placeholder, like median later
    if '/' in str(value):
        return int(str(value).split('/')[1])
    return int(value)

# Apply the function
df['number_of_open_complaints'] = df['number_of_open_complaints'].apply(extract_middle)
df['number_of_open_complaints'] = df['number_of_open_complaints'].astype(int)


# 4. handle null values

In [6]:
# Numerical
for col in df.select_dtypes(include='number').columns:
    df[col].fillna(df[col].median(), inplace=True)

# Categorical
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)


# 5. remove duplicates

In [7]:
# Drop duplicate rows and reset index
df = df.drop_duplicates().reset_index(drop=True)


# save clean data

In [8]:
df.to_csv("cleaned_customer_data.csv", index=False)


# functions (bonus)

In [9]:
# data_cleaning.py
import pandas as pd

def load_data(url):
    return pd.read_csv(url)

def clean_columns(df):
    df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
    df = df.rename(columns={"st": "state"})
    return df

# Add more functions for each step above: clean_values, format_types, handle_nulls, remove_duplicates
